<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Collaborative Filtering From Scratch</b>
</h1>
<div style="font-family:'Times New Roman';">
Time to build user based collaborative filtering myself. The plan is simple, to guess how a user would rate a movie they havent seen, find the users most similiar to them, and take a weighted average of *their* ratings for that movie. Just functions, no fancy library.
</div>

In [1]:
import numpy as np

## A small ratings matrix

Rows are users, columns are movies, and a **0 means the user hasnt rated that movie yet** (those are the ones we want to predict). I made the tastes pretty obvious on purpose, some users clearly love action, some clearly love romance.

In [2]:
users = ['Ayush', 'Akshay', 'Shad', 'Aditya', 'Rohan', 'Karan']
movies = ['Matrix', 'Inception', 'Titanic', 'Notebook', 'Gladiator', 'La La Land']

R = np.array([
    [5, 5, 1, 0, 4, 1],   # Ayush  (action, hasnt seen Notebook)
    [4, 5, 2, 1, 5, 0],   # Akshay (action, hasnt seen La La Land)
    [1, 0, 5, 5, 2, 5],   # Shad   (romance, hasnt seen Inception)
    [2, 1, 4, 5, 0, 4],   # Aditya (romance, hasnt seen Gladiator)
    [5, 4, 0, 2, 5, 1],   # Rohan  (action, hasnt seen Titanic)
    [1, 2, 5, 4, 0, 5],   # Karan  (romance, hasnt seen Gladiator)
], dtype=float)
R.shape

(6, 6)

## Similarity between users

Same cosine similarity as before. Im using the whole rating row (treating the unseen 0s as zeros), wich is a bit rough but fine for a small example.

In [3]:
def cosine(a, b):
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

# how similiar is Ayush (row 0) to everyone else
for v in range(len(users)):
    print(f'Ayush vs {users[v]:7}: {cosine(R[0], R[v]):.3f}')

Ayush vs Ayush  : 1.000
Ayush vs Akshay : 0.964
Ayush vs Shad   : 0.312
Ayush vs Aditya : 0.354
Ayush vs Rohan  : 0.950
Ayush vs Karan  : 0.360


## Predicting a missing rating

To predict user u on movie i, i go through everyone else who *has* rated movie i, weight their rating by how similiar they are to u, and average it. Similar users count more.

In [5]:
def predict_rating(u, i):
    num = 0.0
    den = 0.0
    for v in range(len(users)):
        if v == u or R[v, i] == 0:
            continue
        s = cosine(R[u], R[v])
        num += s * R[v, i]
        den += abs(s)
    return num / den if den > 0 else 0.0

In [6]:
# Ayush hasnt seen Notebook (a romance), so we expect a lowish guess
print('Ayush on Notebook :', round(predict_rating(0, 3), 2))
# Shad hasnt seen Inception (sci-fi), also expect lowish
print('Shad on Inception :', round(predict_rating(2, 1), 2))

Ayush on Notebook : 2.6
Shad on Inception : 2.65


<h2 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Turning it into recommendations</b>
</h2>
<div style="font-family:'Times New Roman';">
Recommending is just predicting every movie the user hasnt seen yet and showing the highest ones. Lets write that and try it on all the users.
</div>

In [8]:
def recommend(u, top_n=3):
    preds = []
    for i in range(len(movies)):
        if R[u, i] == 0:                  # only unseen movies
            preds.append((movies[i], round(float(predict_rating(u, i)), 2)))
    preds.sort(key=lambda x: x[1], reverse=True)
    return preds[:top_n]

for u in range(len(users)):
    print(users[u], 'gets', recommend(u))

Ayush gets [('Notebook', 2.6)]
Akshay gets [('La La Land', 2.39)]
Shad gets [('Inception', 2.65)]
Aditya gets [('Gladiator', 3.49)]
Rohan gets [('Titanic', 2.71)]
Karan gets [('Gladiator', 3.46)]


### quick recap

- user based collaborative filtering guesses a rating from the ratings of similiar users
- cosine similarity decides who counts as similar
- a prediction is just a similarity weighted average of other peoples ratings
- recommending is predicting the unseen movies and taking the top ones

this works but it has to compare against every user every time, and it struggles when the matrix is super empty. Next i'll do matrix factorization wich scales better and ties straight back to SVD.